# Successor Long-Term Phase Plan

This notebook is the inspectable planning companion to `docs/plans/0001_successor_roadmap.md`.
It does two things:

1. shows the current proved state directly from the repo;
2. lays out the planned next phases with explicit goals, acceptance criteria, evidence, and design sketches.

The future-phase code cells are intentionally design-oriented. They print pseudocode or provisional
contract sketches so you can run the notebook and see where the repo is going even before the supporting
implementation exists.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path
import json
import subprocess
import textwrap

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

def git(*args: str) -> str:
    return subprocess.check_output(["git", "-C", str(repo_root), *args], text=True).strip()

def bullet_list(items: tuple[str, ...] | list[str]) -> str:
    return "\n".join(f"- {item}" for item in items)

def heading(title: str) -> None:
    print(f"\n{title}\n" + "=" * len(title))

@dataclass(frozen=True)
class PhasePlan:
    phase_id: str
    title: str
    status: str
    confidence: str
    goal: str
    why_now: tuple[str, ...] = field(default_factory=tuple)
    build: tuple[str, ...] = field(default_factory=tuple)
    acceptance: tuple[str, ...] = field(default_factory=tuple)
    evidence: tuple[str, ...] = field(default_factory=tuple)
    dependencies: tuple[str, ...] = field(default_factory=tuple)
    notes: tuple[str, ...] = field(default_factory=tuple)

def show_phase(phase: PhasePlan) -> None:
    heading(f"Phase {phase.phase_id}: {phase.title}")
    print(f"status: {phase.status}")
    print(f"confidence: {phase.confidence}")
    print(f"goal: {phase.goal}")
    if phase.why_now:
        print("\nwhy now:\n" + bullet_list(phase.why_now))
    if phase.build:
        print("\nbuild:\n" + bullet_list(phase.build))
    if phase.acceptance:
        print("\nacceptance criteria:\n" + bullet_list(phase.acceptance))
    if phase.evidence:
        print("\nacceptance evidence:\n" + bullet_list(phase.evidence))
    if phase.dependencies:
        print("\ndependencies:\n" + bullet_list(phase.dependencies))
    if phase.notes:
        print("\nnotes:\n" + bullet_list(phase.notes))

In [2]:
current_snapshot = {
    "current_commit": git("rev-parse", "--short", "HEAD"),
    "current_branch": git("branch", "--show-current"),
    "tracked_notebooks": sorted(path.name for path in (repo_root / "notebooks").glob("*.ipynb")),
    "prompt_assets": sorted(str(path.relative_to(repo_root)) for path in (repo_root / "prompts").rglob("*.yaml")),
    "test_modules": sorted(str(path.relative_to(repo_root)) for path in (repo_root / "tests").rglob("test_*.py")),
    "roadmap": str((repo_root / "docs" / "plans" / "0001_successor_roadmap.md").relative_to(repo_root)),
    "status_doc": str((repo_root / "docs" / "STATUS.md").relative_to(repo_root)),
}

heading("Repo Snapshot")
print(json.dumps(current_snapshot, indent=2))


Repo Snapshot
{
  "current_commit": "85398ee",
  "current_branch": "main",
  "tracked_notebooks": [
    "01_policy_contracts.ipynb",
    "02_donor_profile_loading.ipynb",
    "03_validation_slice.ipynb",
    "04_review_slice.ipynb",
    "05_review_reporting_slice.ipynb",
    "06_overlay_application_slice.ipynb",
    "07_text_grounded_import_contract.ipynb",
    "08_text_extraction_slice.ipynb",
    "09_successor_long_term_plan.ipynb",
    "10_live_extraction_evaluation.ipynb"
  ],
  "prompt_assets": [
    "prompts/evaluation/judge_candidate_reasonableness.yaml",
    "prompts/extraction/text_to_candidate_assertions.yaml"
  ],
  "test_modules": [
    "tests/evaluation/test_service.py",
    "tests/ontology_runtime/test_loaders.py",
    "tests/ontology_runtime/test_policy.py",
    "tests/ontology_runtime/test_validation.py",
    "tests/pipeline/test_review_service.py",
    "tests/pipeline/test_text_extraction.py"
  ],
  "roadmap": "docs/plans/0001_successor_roadmap.md",
  "status_doc": "d

In [3]:
completed_phases = [
    PhasePlan(
        phase_id="0",
        title="Ontology Runtime Slice",
        status="completed",
        confidence="high",
        goal="Prove ontology/runtime contracts, donor loading, policy semantics, and local validation.",
        evidence=(
            "tests/ontology_runtime/",
            "notebooks/01_policy_contracts.ipynb",
            "notebooks/02_donor_profile_loading.ipynb",
            "notebooks/03_validation_slice.ipynb",
        ),
    ),
    PhasePlan(
        phase_id="1",
        title="Pipeline Review Slice",
        status="completed",
        confidence="high",
        goal="Persist validation, proposals, and configurable mixed-mode review state.",
        evidence=(
            "tests/pipeline/test_review_service.py",
            "notebooks/04_review_slice.ipynb",
        ),
    ),
    PhasePlan(
        phase_id="2",
        title="First User-Visible Capability",
        status="completed",
        confidence="high",
        goal="Recover governed review of candidate assertions as the first actual payoff.",
        evidence=(
            "src/onto_canon6/surfaces/review_report.py",
            "notebooks/05_review_reporting_slice.ipynb",
        ),
    ),
    PhasePlan(
        phase_id="3",
        title="Overlay Application and Query Surface",
        status="completed",
        confidence="high",
        goal="Make accepted ontology-growth decisions operational and inspectable.",
        evidence=(
            "src/onto_canon6/pipeline/overlay_service.py",
            "notebooks/06_overlay_application_slice.ipynb",
        ),
    ),
    PhasePlan(
        phase_id="4",
        title="Text-Grounded Producer Integration",
        status="completed",
        confidence="high",
        goal="Route raw text through llm_client into evidence-grounded candidate assertions.",
        evidence=(
            "src/onto_canon6/pipeline/text_extraction.py",
            "prompts/extraction/text_to_candidate_assertions.yaml",
            "notebooks/07_text_grounded_import_contract.ipynb",
            "notebooks/08_text_extraction_slice.ipynb",
        ),
    ),
    PhasePlan(
        phase_id="5",
        title="Live Extraction Evaluation and Quality Harness",
        status="completed",
        confidence="high",
        goal="Measure real extraction usefulness before expanding the workflow further.",
        evidence=(
            "src/onto_canon6/evaluation/service.py",
            "prompts/evaluation/judge_candidate_reasonableness.yaml",
            "tests/evaluation/test_service.py",
            "notebooks/10_live_extraction_evaluation.ipynb",
            "docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md",
        ),
        notes=(
            "This phase is now complete and is the guardrail for future extraction work.",
        ),
    ),
]

planned_phases = [
    PhasePlan(
        phase_id="6",
        title="First Operational Surface",
        status="planned",
        confidence="medium-high",
        goal="Make the workflow usable without direct Python or notebook editing.",
        why_now=(
            "The underlying services are now stable enough to expose thin handlers.",
            "CLI is the simplest surface that can prove operability without framework drag.",
        ),
        build=(
            "Thin CLI for extract, list, review, and apply actions.",
            "JSON output mode for scripting.",
            "Minimal integration tests and one transcript/notebook proof.",
        ),
        acceptance=(
            "A user can go from raw text to persisted candidate assertions through the CLI.",
            "A user can inspect candidates/proposals and record review decisions through the CLI.",
            "Handlers delegate to services rather than owning workflow logic.",
        ),
        evidence=(
            "CLI integration tests.",
            "One end-to-end CLI transcript or notebook.",
        ),
        dependencies=("Phase 5 evaluation guardrails",),
        notes=("This is now the locked next phase.",),
    ),
    PhasePlan(
        phase_id="7",
        title="Domain Pack Generalization",
        status="planned",
        confidence="medium",
        goal="Prove the same architecture supports a second real domain pack.",
        why_now=(
            "A second pack is the cleanest test that the current boundaries are real.",
            "DoDAF is useful here precisely because it is different and still deferred.",
        ),
        build=(
            "Second pack with at least one strict and one mixed profile.",
            "Pack-local notebook proof for validation, review, and overlays.",
            "Any pack-specific extraction context without core branching.",
        ),
        acceptance=(
            "Second pack loads through existing pack/profile machinery.",
            "Same review and overlay workflow works for the second pack.",
            "No domain-specific logic is added to core modules.",
        ),
        evidence=(
            "Second-pack loading/validation tests.",
            "One notebook showing second-pack workflow behavior.",
        ),
        dependencies=("Phase 6 operational surface",),
        notes=("DoDAF is the recommended exemplar here, not earlier.",),
    ),
    PhasePlan(
        phase_id="8",
        title="Artifact Lineage Recovery",
        status="planned",
        confidence="medium",
        goal="Recover artifact-backed provenance from v1 without rebuilding a fused runtime.",
        why_now=(
            "Artifact lineage is one of the strongest donor ideas from v1.",
            "It becomes more valuable once there is an operational surface and multiple packs.",
        ),
        build=(
            "Typed artifact reference model for raw, derived, and analysis artifacts.",
            "Artifact-to-candidate linkage plus provenance traversal.",
            "One small workflow showing a claim supported by an analysis artifact rather than only raw text.",
        ),
        acceptance=(
            "Artifacts can be registered and linked without moving domain logic into core.",
            "A candidate assertion can cite an artifact-backed support path.",
            "Lineage remains inspectable in reports and tests.",
        ),
        evidence=(
            "Artifact-lineage tests.",
            "One notebook showing artifact-backed provenance.",
        ),
        dependencies=("Phase 6 operational surface", "Phase 7 second-pack proof"),
    ),
    PhasePlan(
        phase_id="9",
        title="Epistemic Extension",
        status="planned",
        confidence="medium",
        goal="Add optional epistemic reasoning without collapsing subsystem boundaries.",
        why_now=(
            "Epistemic reasoning is valuable, but only once the assertion/review/provenance base is stable.",
            "The extension must remain optional rather than silently turning into core policy.",
        ),
        build=(
            "Extension-local epistemic models and operators.",
            "Integration with candidate/review outcomes where appropriate.",
            "Notebook proof showing contradiction/tension style reasoning without contaminating core runtime.",
        ),
        acceptance=(
            "Epistemic logic lives outside core modules.",
            "The base workflow still works with the extension disabled.",
            "The extension has its own tests and notebook proof.",
        ),
        evidence=(
            "Epistemic extension tests.",
            "One notebook showing extension behavior.",
        ),
        dependencies=("Phase 8 artifact lineage",),
    ),
    PhasePlan(
        phase_id="10",
        title="Product-Facing Workflow Integration",
        status="planned",
        confidence="medium-low",
        goal="Connect the proved stack into one real end-to-end workflow with user-visible leverage.",
        why_now=(
            "The lineage only matters if the system becomes useful to a real workflow again.",
            "This phase should happen only after the underlying surfaces are no longer notebook-only.",
        ),
        build=(
            "Choose one real workflow and one real producer/consumer path.",
            "Wire extraction, review, overlays, provenance, and reporting together without bypasses.",
            "Document what user-visible leverage the integrated workflow actually delivers.",
        ),
        acceptance=(
            "A user can complete one real workflow end to end without ad hoc Python editing.",
            "The workflow shows clear leverage over the pre-successor state.",
            "The workflow does not reintroduce a fused runtime blob.",
        ),
        evidence=(
            "One end-to-end demo notebook or transcript.",
            "One short design note explaining why this workflow was chosen.",
        ),
        dependencies=("Phase 6 operational surface", "Phase 8 artifact lineage", "Phase 9 epistemic extension (if needed)"),
    ),
]

heading("Completed Phases")
for phase in completed_phases:
    print(f"Phase {phase.phase_id}: {phase.title} [{phase.status}]")
    print(f"  goal: {phase.goal}")
    print(f"  evidence count: {len(phase.evidence)}")

heading("Planned Phases")
for phase in planned_phases:
    print(f"Phase {phase.phase_id}: {phase.title} [{phase.status}] confidence={phase.confidence}")
    print(f"  goal: {phase.goal}")
    if phase.dependencies:
        print(f"  dependencies: {', '.join(phase.dependencies)}")
    if phase.notes:
        print(f"  notes: {' '.join(phase.notes)}")
    print()




Completed Phases
Phase 0: Ontology Runtime Slice [completed]
  goal: Prove ontology/runtime contracts, donor loading, policy semantics, and local validation.
  evidence count: 4
Phase 1: Pipeline Review Slice [completed]
  goal: Persist validation, proposals, and configurable mixed-mode review state.
  evidence count: 2
Phase 2: First User-Visible Capability [completed]
  goal: Recover governed review of candidate assertions as the first actual payoff.
  evidence count: 2
Phase 3: Overlay Application and Query Surface [completed]
  goal: Make accepted ontology-growth decisions operational and inspectable.
  evidence count: 2
Phase 4: Text-Grounded Producer Integration [completed]
  goal: Route raw text through llm_client into evidence-grounded candidate assertions.
  evidence count: 4
Phase 5: Live Extraction Evaluation and Quality Harness [completed]
  goal: Measure real extraction usefulness before expanding the workflow further.
  evidence count: 5

Planned Phases
Phase 6: First Op

## Recommended Order

The sequence below is the actual recommended path, not just a bag of ideas.
Phase 6 is now the locked next phase. Later phases are still planned, but their
detailed contracts should only be implemented when their predecessor has passed its exit criteria.


In [4]:
recommended_sequence = [
    ("6", "Expose the proved workflow through a CLI, not direct Python calls."),
    ("7", "Prove a second pack such as DoDAF without core changes."),
    ("8", "Recover artifact lineage once multiple workflows need provenance depth."),
    ("9", "Add epistemic reasoning as an extension, not hidden policy."),
    ("10", "Deliver one real product-facing workflow once the stack is actually ready."),
]

heading("Recommended Sequence")
for phase_id, rationale in recommended_sequence:
    print(f"Phase {phase_id}: {rationale}")



Recommended Sequence
Phase 6: Expose the proved workflow through a CLI, not direct Python calls.
Phase 7: Prove a second pack such as DoDAF without core changes.
Phase 8: Recover artifact lineage once multiple workflows need provenance depth.
Phase 9: Add epistemic reasoning as an extension, not hidden policy.
Phase 10: Deliver one real product-facing workflow once the stack is actually ready.


## Phase 5 Completed Slice


In [5]:
phase = next(item for item in completed_phases if item.phase_id == '5')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))



Phase 5: Live Extraction Evaluation and Quality Harness
status: completed
confidence: high
goal: Measure real extraction usefulness before expanding the workflow further.

acceptance evidence:
- src/onto_canon6/evaluation/service.py
- prompts/evaluation/judge_candidate_reasonableness.yaml
- tests/evaluation/test_service.py
- notebooks/10_live_extraction_evaluation.ipynb
- docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md

notes:
- This phase is now complete and is the guardrail for future extraction work.

as_json:
{
  "phase_id": "5",
  "title": "Live Extraction Evaluation and Quality Harness",
  "status": "completed",
  "confidence": "high",
  "goal": "Measure real extraction usefulness before expanding the workflow further.",
  "why_now": [],
  "build": [],
  "acceptance": [],
  "evidence": [
    "src/onto_canon6/evaluation/service.py",
    "prompts/evaluation/judge_candidate_reasonableness.yaml",
    "tests/evaluation

In [6]:
implemented_assets = (
    "src/onto_canon6/evaluation/models.py",
    "src/onto_canon6/evaluation/service.py",
    "prompts/evaluation/judge_candidate_reasonableness.yaml",
    "tests/evaluation/test_service.py",
    "tests/fixtures/psyop_eval_slice.json",
    "notebooks/10_live_extraction_evaluation.ipynb",
    "docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md",
)
heading("Implemented Slice")
print(bullet_list(implemented_assets))



Implemented Slice
- src/onto_canon6/evaluation/models.py
- src/onto_canon6/evaluation/service.py
- prompts/evaluation/judge_candidate_reasonableness.yaml
- tests/evaluation/test_service.py
- tests/fixtures/psyop_eval_slice.json
- notebooks/10_live_extraction_evaluation.ipynb
- docs/adr/0005-separate-live-extraction-reasonableness-from-structural-validation-and-canonicalization-fidelity.md


## Phase 6 Detail

In [7]:
phase = next(item for item in planned_phases if item.phase_id == '6')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))


Phase 6: First Operational Surface
status: planned
confidence: medium-high
goal: Make the workflow usable without direct Python or notebook editing.

why now:
- The underlying services are now stable enough to expose thin handlers.
- CLI is the simplest surface that can prove operability without framework drag.

build:
- Thin CLI for extract, list, review, and apply actions.
- JSON output mode for scripting.
- Minimal integration tests and one transcript/notebook proof.

acceptance criteria:
- A user can go from raw text to persisted candidate assertions through the CLI.
- A user can inspect candidates/proposals and record review decisions through the CLI.
- Handlers delegate to services rather than owning workflow logic.

acceptance evidence:
- CLI integration tests.
- One end-to-end CLI transcript or notebook.

dependencies:
- Phase 5 evaluation guardrails

notes:
- This is now the locked next phase.

as_json:
{
  "phase_id": "6",
  "title": "First Operational Surface",
  "status": 

In [8]:
pseudocode = textwrap.dedent(r"""
# Proposed files:
# - src/onto_canon6/cli.py
# - tests/integration/test_cli_flow.py

@app.command("extract-text")
def extract_text(source_path: Path, profile: str, output: Literal["table", "json"] = "table") -> None:
    # thin handler only
    # delegates to TextExtractionService + ReviewService
    ...

@app.command("review-candidate")
def review_candidate(candidate_id: str, decision: Literal["accept", "reject"], note: str | None = None) -> None:
    # delegates to ReviewService.review_candidate(...)
    ...

@app.command("apply-overlay")
def apply_overlay(proposal_id: str, actor_id: str) -> None:
    # delegates to OverlayApplicationService.apply(...)
    ...
""").strip()
heading("Pseudocode Sketch")
print(pseudocode)


Pseudocode Sketch
# Proposed files:
# - src/onto_canon6/cli.py
# - tests/integration/test_cli_flow.py

@app.command("extract-text")
def extract_text(source_path: Path, profile: str, output: Literal["table", "json"] = "table") -> None:
    # thin handler only
    # delegates to TextExtractionService + ReviewService
    ...

@app.command("review-candidate")
def review_candidate(candidate_id: str, decision: Literal["accept", "reject"], note: str | None = None) -> None:
    # delegates to ReviewService.review_candidate(...)
    ...

@app.command("apply-overlay")
def apply_overlay(proposal_id: str, actor_id: str) -> None:
    # delegates to OverlayApplicationService.apply(...)
    ...


## Phase 7 Detail

In [9]:
phase = next(item for item in planned_phases if item.phase_id == '7')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))


Phase 7: Domain Pack Generalization
status: planned
confidence: medium
goal: Prove the same architecture supports a second real domain pack.

why now:
- A second pack is the cleanest test that the current boundaries are real.
- DoDAF is useful here precisely because it is different and still deferred.

build:
- Second pack with at least one strict and one mixed profile.
- Pack-local notebook proof for validation, review, and overlays.
- Any pack-specific extraction context without core branching.

acceptance criteria:
- Second pack loads through existing pack/profile machinery.
- Same review and overlay workflow works for the second pack.
- No domain-specific logic is added to core modules.

acceptance evidence:
- Second-pack loading/validation tests.
- One notebook showing second-pack workflow behavior.

dependencies:
- Phase 6 operational surface

notes:
- DoDAF is the recommended exemplar here, not earlier.

as_json:
{
  "phase_id": "7",
  "title": "Domain Pack Generalization",
  "

In [10]:
pseudocode = textwrap.dedent(r"""
# Proposed pack layout:
# domain_packs/dodaf/
#   manifests/
#   profiles/
#   sample_sources/
#   notebooks/

second_pack_plan = {
    "pack_id": "dodaf",
    "profiles": ["dodaf_closed", "dodaf_mixed"],
    "proof": [
        "load pack",
        "validate candidate assertions",
        "route unknown items to overlay proposals",
        "apply accepted additions",
    ],
}

# No core branching like:
# if profile_id.startswith("dodaf"):
#     ...
# All variation should come from pack/profile data.
""").strip()
heading("Pseudocode Sketch")
print(pseudocode)


Pseudocode Sketch
# Proposed pack layout:
# domain_packs/dodaf/
#   manifests/
#   profiles/
#   sample_sources/
#   notebooks/

second_pack_plan = {
    "pack_id": "dodaf",
    "profiles": ["dodaf_closed", "dodaf_mixed"],
    "proof": [
        "load pack",
        "validate candidate assertions",
        "route unknown items to overlay proposals",
        "apply accepted additions",
    ],
}

# No core branching like:
# if profile_id.startswith("dodaf"):
#     ...
# All variation should come from pack/profile data.


## Phase 8 Detail

In [11]:
phase = next(item for item in planned_phases if item.phase_id == '8')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))


Phase 8: Artifact Lineage Recovery
status: planned
confidence: medium
goal: Recover artifact-backed provenance from v1 without rebuilding a fused runtime.

why now:
- Artifact lineage is one of the strongest donor ideas from v1.
- It becomes more valuable once there is an operational surface and multiple packs.

build:
- Typed artifact reference model for raw, derived, and analysis artifacts.
- Artifact-to-candidate linkage plus provenance traversal.
- One small workflow showing a claim supported by an analysis artifact rather than only raw text.

acceptance criteria:
- Artifacts can be registered and linked without moving domain logic into core.
- A candidate assertion can cite an artifact-backed support path.
- Lineage remains inspectable in reports and tests.

acceptance evidence:
- Artifact-lineage tests.
- One notebook showing artifact-backed provenance.

dependencies:
- Phase 6 operational surface
- Phase 7 second-pack proof

as_json:
{
  "phase_id": "8",
  "title": "Artifact Li

In [12]:
pseudocode = textwrap.dedent(r"""
# Proposed files:
# - src/onto_canon6/artifacts/models.py
# - src/onto_canon6/artifacts/store.py
# - src/onto_canon6/surfaces/lineage_report.py

class ArtifactRef(BaseModel):
    artifact_id: str
    artifact_kind: Literal["source", "derived", "analysis"]
    uri: str
    label: str | None = None
    metadata: dict[str, JSONValue] = {}

class AssertionArtifactLink(BaseModel):
    assertion_id: str
    artifact_id: str
    relationship: Literal["quoted_from", "derived_from", "supported_by_analysis"]
    detail: str | None = None

def build_lineage_report(assertion_id: str) -> LineageReport:
    # show raw source, derived artifact, analysis artifact, and review provenance
    ...
""").strip()
heading("Pseudocode Sketch")
print(pseudocode)


Pseudocode Sketch
# Proposed files:
# - src/onto_canon6/artifacts/models.py
# - src/onto_canon6/artifacts/store.py
# - src/onto_canon6/surfaces/lineage_report.py

class ArtifactRef(BaseModel):
    artifact_id: str
    artifact_kind: Literal["source", "derived", "analysis"]
    uri: str
    label: str | None = None
    metadata: dict[str, JSONValue] = {}

class AssertionArtifactLink(BaseModel):
    assertion_id: str
    artifact_id: str
    relationship: Literal["quoted_from", "derived_from", "supported_by_analysis"]
    detail: str | None = None

def build_lineage_report(assertion_id: str) -> LineageReport:
    # show raw source, derived artifact, analysis artifact, and review provenance
    ...


## Phase 9 Detail

In [13]:
phase = next(item for item in planned_phases if item.phase_id == '9')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))


Phase 9: Epistemic Extension
status: planned
confidence: medium
goal: Add optional epistemic reasoning without collapsing subsystem boundaries.

why now:
- Epistemic reasoning is valuable, but only once the assertion/review/provenance base is stable.
- The extension must remain optional rather than silently turning into core policy.

build:
- Extension-local epistemic models and operators.
- Integration with candidate/review outcomes where appropriate.
- Notebook proof showing contradiction/tension style reasoning without contaminating core runtime.

acceptance criteria:
- Epistemic logic lives outside core modules.
- The base workflow still works with the extension disabled.
- The extension has its own tests and notebook proof.

acceptance evidence:
- Epistemic extension tests.
- One notebook showing extension behavior.

dependencies:
- Phase 8 artifact lineage

as_json:
{
  "phase_id": "9",
  "title": "Epistemic Extension",
  "status": "planned",
  "confidence": "medium",
  "goal": 

In [14]:
pseudocode = textwrap.dedent(r"""
# Proposed files:
# - src/onto_canon6/extensions/epistemic/models.py
# - src/onto_canon6/extensions/epistemic/service.py

class EpistemicState(BaseModel):
    assertion_id: str
    confidence: float | None = None
    status: Literal["active", "weakened", "superseded", "retracted"]
    rationale: str | None = None

def supersede_assertion(old_assertion_id: str, new_assertion_id: str, actor_id: str) -> None:
    # extension-owned operator
    # records relation without changing core assertion storage semantics
    ...
""").strip()
heading("Pseudocode Sketch")
print(pseudocode)


Pseudocode Sketch
# Proposed files:
# - src/onto_canon6/extensions/epistemic/models.py
# - src/onto_canon6/extensions/epistemic/service.py

class EpistemicState(BaseModel):
    assertion_id: str
    confidence: float | None = None
    status: Literal["active", "weakened", "superseded", "retracted"]
    rationale: str | None = None

def supersede_assertion(old_assertion_id: str, new_assertion_id: str, actor_id: str) -> None:
    # extension-owned operator
    # records relation without changing core assertion storage semantics
    ...


## Phase 10 Detail

In [15]:
phase = next(item for item in planned_phases if item.phase_id == '10')
show_phase(phase)

print("\nas_json:")
print(json.dumps(asdict(phase), indent=2))


Phase 10: Product-Facing Workflow Integration
status: planned
confidence: medium-low
goal: Connect the proved stack into one real end-to-end workflow with user-visible leverage.

why now:
- The lineage only matters if the system becomes useful to a real workflow again.
- This phase should happen only after the underlying surfaces are no longer notebook-only.

build:
- Choose one real workflow and one real producer/consumer path.
- Wire extraction, review, overlays, provenance, and reporting together without bypasses.
- Document what user-visible leverage the integrated workflow actually delivers.

acceptance criteria:
- A user can complete one real workflow end to end without ad hoc Python editing.
- The workflow shows clear leverage over the pre-successor state.
- The workflow does not reintroduce a fused runtime blob.

acceptance evidence:
- One end-to-end demo notebook or transcript.
- One short design note explaining why this workflow was chosen.

dependencies:
- Phase 6 operation

In [16]:
pseudocode = textwrap.dedent(r"""
# Proposed product slice:
# 1. ingest selected source texts
# 2. extract candidate assertions
# 3. review/accept/reject candidates and proposals
# 4. apply ontology additions where approved
# 5. query/export governed assertions with provenance

workflow_demo = {
    "input": "real source text or corpus slice",
    "surface": "CLI first, then possibly MCP",
    "outputs": [
        "review report",
        "accepted assertions",
        "overlay additions",
        "provenance/lineage summary",
    ],
}

# The success test is not just 'did the code run?'
# It is 'did the workflow save real analyst/operator time?'
""").strip()
heading("Pseudocode Sketch")
print(pseudocode)


Pseudocode Sketch
# Proposed product slice:
# 1. ingest selected source texts
# 2. extract candidate assertions
# 3. review/accept/reject candidates and proposals
# 4. apply ontology additions where approved
# 5. query/export governed assertions with provenance

workflow_demo = {
    "input": "real source text or corpus slice",
    "surface": "CLI first, then possibly MCP",
    "outputs": [
        "review report",
        "accepted assertions",
        "overlay additions",
        "provenance/lineage summary",
    ],
}

# The success test is not just 'did the code run?'
# It is 'did the workflow save real analyst/operator time?'


## Decision Gates

These are the main questions that should be answered explicitly at the phase boundaries,
not buried in ad hoc implementation drift.

In [17]:
decision_gates = {
    "benchmark_follow_on": [
        "How large should the benchmark corpus become before we treat its aggregate numbers as stronger evidence?",
        "Should structurally invalid but text-supported candidates get a dedicated remediation report before broader benchmark growth?",
    ],
    "phase_6": [
        "Does CLI remain sufficient, or is there real pressure for MCP/UI after the first operational proof?",
    ],
    "phase_7": [
        "Is DoDAF still the right second-pack exemplar once Phase 6 is real?",
    ],
    "phase_8": [
        "How much artifact lineage is enough before it becomes another oversized subsystem?",
    ],
    "phase_10": [
        "What single workflow best demonstrates real user-visible leverage?",
    ],
}

heading("Decision Gates")
print(json.dumps(decision_gates, indent=2))



Decision Gates
{
  "benchmark_follow_on": [
    "How large should the benchmark corpus become before we treat its aggregate numbers as stronger evidence?",
    "Should structurally invalid but text-supported candidates get a dedicated remediation report before broader benchmark growth?"
  ],
  "phase_6": [
    "Does CLI remain sufficient, or is there real pressure for MCP/UI after the first operational proof?"
  ],
  "phase_7": [
    "Is DoDAF still the right second-pack exemplar once Phase 6 is real?"
  ],
  "phase_8": [
    "How much artifact lineage is enough before it becomes another oversized subsystem?"
  ],
  "phase_10": [
    "What single workflow best demonstrates real user-visible leverage?"
  ]
}
